In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

import numpy as np

import channel_design
from channel_design import (
    annualised_cost,
    candidate_designs,
    route_with_head_loss_iteration,
    start_map,
)

In [3]:
#plain the model scans
# m = start_map()
# m

In [ ]:
## 1. SETTINGS ##


# ---- Paths ----
PROJECT_DIR = '../elliot_main_channel'
ROUTES_DIR = os.path.join(PROJECT_DIR, '0_run1')
SEQUENTIAL_RUN_TEMPLATE = os.path.join('inputs/Sequential_Run.csv')

DEM_PATH = "inputs/DEM/merged_dem.tif"
ROCK_DEPTH_PATH = "inputs/DEM/low_res_resampled.tif"

# ---- Routing area (lon/lat): left, bottom, right, top ----
ROUTING_BOUNDS = (147, -21, 149, -19)

# ---- Channel hydraulics ----
MANNING_N = 0.03
SIDE_SLOPE = 2  # horizontal : vertical


# ---- Sequential run settings ----
MAX_RUNS = 4
RESIDUAL_HEAD_LOSS_TOL_M = 0.1

# ---- Per-section inputs ----
SECTIONS = {
    'section1': dict(
        start=(147.31, -19.81),
        finish=(147.857, -19.978),       # before pumping station
        start_elevation=36,              # DFL
        gross_diversions_m3_s=19.4,
        default_linear_cost=583930,
        total_loss_GL=8.7,
        approximate_length_km=67,        # used for the length penalty
        no_cells=1991,
    ),
    'section2': dict(
        start=(147.866, -19.974),        # after pumping station
        finish=(148.079, -20.086),       # before Euri Creek
        start_elevation=40,              # DFL
        gross_diversions_m3_s=10,
        default_linear_cost=577045,
        total_loss_GL=2.9,
        approximate_length_km=40,
        no_cells=805,
    ),
}
# ---- Router settings ----
# SAMPLE_F = 250
# MAX_FILL_SIPHON_ENDS = 5
# Pass the settings above to channel_design
channel_design.configure(
    ROUTES_DIR=ROUTES_DIR,
    SEQUENTIAL_RUN_TEMPLATE=SEQUENTIAL_RUN_TEMPLATE,
    DEM_PATH=DEM_PATH,
    ROCK_DEPTH_PATH=ROCK_DEPTH_PATH,
    ROUTING_BOUNDS=ROUTING_BOUNDS,
    MANNING_N=MANNING_N,
    SIDE_SLOPE=SIDE_SLOPE,
    SAMPLE_F=250,
    MAX_FILL_SIPHON_ENDS=5,
    MAX_RUNS=MAX_RUNS,
    RESIDUAL_HEAD_LOSS_TOL_M=RESIDUAL_HEAD_LOSS_TOL_M,
    SECTIONS=SECTIONS,
)


In [ ]:
## 2. CANDIDATE DESIGNS ##

# Flow velocities (m/s) swept when generating candidate channel designs
VELOCITY_RANGES = {
    'section1': np.arange(0.490, 0.491, 0.01),
    'section2': np.arange(0.425, 0.444, 0.001),
}

version = 'cost-sensitivity-analysis'
section = 'section2'

designs = candidate_designs(section, ratios=[5], velocities=VELOCITY_RANGES[section])
[design[5] for design in designs]  # velocities kept

[np.float64(0.425),
 np.float64(0.427),
 np.float64(0.429),
 np.float64(0.431),
 np.float64(0.433),
 np.float64(0.435),
 np.float64(0.437),
 np.float64(0.439),
 np.float64(0.441),
 np.float64(0.444)]

In [ ]:
## 3. run WaterRoute for the candidate designs ##
for section, ratio, depth, width, slope_1_in_x, velocity in designs:
    run_name = f'{version}-{section}-r{ratio}-d{depth}-w{width}-S{slope_1_in_x}-v{velocity:.3f}'
    work_dir = os.path.join(ROUTES_DIR, run_name)
    if os.path.isdir(work_dir):
        print("Exists:", work_dir)
        continue
    print(work_dir)
    route_with_head_loss_iteration(number_of_runs=3, width_to_depth_ratio=ratio, water_depth=depth, bed_width=width,
                                   run_name=run_name, section=section, slope_for_channel_dimension=1 / slope_1_in_x)

Exists: run1\1_Routes\v22-section1-r5-d2.38-w11.89-S9814-v0.490


In [ ]:

## CANDIDATE DESIGNS  for sensitivity analysis ##

# Flow velocities (m/s) swept when generating candidate channel designs
VELOCITY_RANGES = {
    'section1': np.arange(0.490, 0.491, 0.01),
    'section2': np.arange(0.425, 0.444, 0.001),
}

version = 'cost-sensitivity-analysis'
section = 'section2'

designs = candidate_designs(section, ratios=[5], velocities=VELOCITY_RANGES[section])
[design[5] for design in designs]  # velocities kept

## sensitivity analysis for section2 linear cost ##

COST_FACTORS = [0.7, 1.3]          # 70% and 130%; add 1.0 if you want a baseline run
BASE_COST_S2 = 577045              # baseline default_linear_cost for section2

for cost_factor in COST_FACTORS:
    SECTIONS['section2']['default_linear_cost'] = SECTIONS['section2']['default_linear_cost']* cost_factor
    SECTIONS['section2']['total_loss_GL'] = SECTIONS['section2']['total_loss_GL']* (-cost_factor)  # scale the total loss by the same factor

    # Re-pass settings so channel_design uses the updated cost
    channel_design.configure(
        ROUTES_DIR=ROUTES_DIR,
        SEQUENTIAL_RUN_TEMPLATE=SEQUENTIAL_RUN_TEMPLATE,
        DEM_PATH=DEM_PATH,
        ROCK_DEPTH_PATH=ROCK_DEPTH_PATH,
        ROUTING_BOUNDS=ROUTING_BOUNDS,
        MANNING_N=MANNING_N,
        SIDE_SLOPE=SIDE_SLOPE,
        SAMPLE_F=250,
        MAX_FILL_SIPHON_ENDS=5,
        MAX_RUNS=MAX_RUNS,
        RESIDUAL_HEAD_LOSS_TOL_M=RESIDUAL_HEAD_LOSS_TOL_M,
        SECTIONS=SECTIONS,
    )

    for section, ratio, depth, width, slope_1_in_x, velocity in designs:
        run_name = (f'{version}-{section}-r{ratio}-d{depth}-w{width}'
                    f'-S{slope_1_in_x}-v{velocity:.3f}-c{int(round(cost_factor * 100))}')
        work_dir = os.path.join(ROUTES_DIR, run_name)
        if os.path.isdir(work_dir):
            print("Exists:", work_dir)
            continue
        print(work_dir, "| section2 linear cost =", SECTIONS['section2']['default_linear_cost'])
        route_with_head_loss_iteration(
            number_of_runs=3, width_to_depth_ratio=ratio, water_depth=depth,
            bed_width=width, run_name=run_name, section=section,
            slope_for_channel_dimension=1 / slope_1_in_x)

# Put the baseline cost back so later cells aren't affected
SECTIONS['section2']['default_linear_cost'] = BASE_COST_S2
channel_design.configure(SECTIONS=SECTIONS) if False else None  # see note below

run1\1_Routes\cost-sensitivity-analysis-section2-r5-d1.83-w9.17-S9225-v0.425-c70 | section2 linear cost = 403932
Calculated channel dimensions: slope = -0.00010840108401084011, ratio_width_depth = 5, water depth = 1.83 m, bed width = 9.17 m


c:\Users\seo010\waterRoute\waterRoute\waterroute\channelroute.py:518: RuntimeWarning: invalid value encountered in cast
  rock_i = np.rint(rockDepth_raster).astype(np.int16)
c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'cost-sensitivity-analysis-section2-r5-d1.83-w9.17-S9225-v0.425-c70_40' adjusted to 'cost_sensitivity_analysis_section2_r5_d1.83_w9.17_S9225_v0.425_c70_40' for XML validity.
  ogr_write(


length of 40 is 30.67km. Cost is (2020 AUD) M$8.42
0


c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'cost-sensitivity-analysis-section2-r5-d1.83-w9.17-S9225-v0.425-c70_40_cross_drainage' adjusted to 'cost_sensitivity_analysis_section2_r5_d1.83_w9.17_S9225_v0.425_c70_40_cross_drainage' for XML validity.
  ogr_write(
c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'cost-sensitivity-analysis-section2-r5-d1.83-w9.17-S9225-v0.425-c70_40_cross_drainage_catchment' adjusted to 'cost_sensitivity_analysis_section2_r5_d1.83_w9.17_S9225_v0.425_c70_40_cross_drainage_catchment' for XML validity.
  ogr_write(


Calculated channel dimensions: slope = -0.00010840108401084011, ratio_width_depth = 5, water depth = 2.06 m, bed width = 10.30 m


c:\Users\seo010\waterRoute\waterRoute\waterroute\channelroute.py:518: RuntimeWarning: invalid value encountered in cast
  rock_i = np.rint(rockDepth_raster).astype(np.int16)
c:\Users\seo010\miniforge3\envs\channelroute\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Layer name 'cost-sensitivity-analysis-section2-r5-d1.83-w9.17-S9225-v0.425-c70_40' adjusted to 'cost_sensitivity_analysis_section2_r5_d1.83_w9.17_S9225_v0.425_c70_40' for XML validity.
  ogr_write(
